Nombre: Felipe Aravena 
Rut: 21.128.400-5
Fecha: 03-06-2026

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

In [24]:
#1 Ejercicio 1: Inspección inicial de archivos 
#1.1
ruta_enero = Path('datos/ventas/ventas_enero.csv')

df_enero = pd.read_csv('ventas_enero.csv', parse_dates=["fecha"])
df_enero.head()

,fecha,producto,vendedor,region,cantidad,precio_unitario,total
0,2024-01-01,Hub USB,Matias Soto,Poniente,1,18000,18000
1,2024-01-02,Mouse,Matias Soto,Norte,4,22000,88000
2,2024-01-03,Auriculares,Matias Soto,Poniente,2,68000,136000
3,2024-01-04,Hub USB,Matias Soto,Poniente,5,18000,90000
4,2024-01-05,Teclado,Paula Mena,Norte,4,45000,180000


In [33]:
#1.2

print(f'dimensiones del dataframe \n{df_enero.shape}')
print(f'\ntipo de dato por columna \n{df_enero.dtypes}')
print(f'\nvalores nulos por columna \n{df_enero.isnull().sum()}') 
print(f'\nestadisticas descriptivas \n{df_enero.describe()}') 

#Filtrar registros donde vendedor no sea nulo
df_enero_filtrado = df_enero[df_enero['vendedor'].notna()]
print(f"\nFilas originales: {len(df_enero)}  Filas tras filtrar nulos en vendedor: {len(df_enero_filtrado)}")

dimensiones del dataframe 
(31, 7)

tipo de dato por columna 
fecha              datetime64[ns]
producto                   object
vendedor                   object
region                     object
cantidad                    int64
precio_unitario             int64
total                       int64
dtype: object

valores nulos por columna 
fecha              0
producto           0
vendedor           1
region             0
cantidad           0
precio_unitario    0
total              0
dtype: int64

estadisticas descriptivas 
                     fecha   cantidad  precio_unitario         total
count                   31  31.000000        31.000000  3.100000e+01
mean   2024-01-16 00:00:00   2.838710     73483.870968  1.851290e+05
min    2024-01-01 00:00:00   1.000000     18000.000000  1.800000e+04
25%    2024-01-08 12:00:00   2.000000     33500.000000  7.500000e+04
50%    2024-01-16 00:00:00   3.000000     55000.000000  1.350000e+05
75%    2024-01-23 12:00:00   4.000000     75000.000000  

Se importó el archivo individual ventas_enero.csv usando la librería pathlib y se transformó la columna cronológica a tipo datetime. Se ejecutó una inspección del dataset para caracterizar las variables numéricas y detectar nulos en la columna vendedor. Finalmente, se aplicó un filtro lógico con .notna() para remover las filas vacías de dicha columna.

In [75]:
#Ejercicio 2: Consolidacion por lotes con glob y pd.concat
#2.1
ruta_ventas = Path('datos/ventas')
archivos_csv = sorted(list(ruta_ventas.glob('*.csv')))

print(f"Archivos encontrados en la carpeta: {len(archivos_csv)}")

# Listas y contadores para el pipeline de consolidación
dataframes = []
total_filas_individuales = 0

for archivo in archivos_csv:
    df_temp = pd.read_csv(archivo, parse_dates=['fecha'])
    
    # Extraer el nombre del mes desde el nombre del archivo 
    mes = archivo.stem.split('_')[1]
    # Agregar la nueva columna con el mes
    df_temp['mes'] = mes
    # Sumar el tamaño del archivo actual al contador de validación
    total_filas_individuales += len(df_temp)

    dataframes.append(df_temp)

# 3. Consolidar todos los archivos en un único DataFrame global
df_consolidado = pd.concat(dataframes, ignore_index=True)
print(df_consolidado.shape)
# 4. Validación matemática obligatoria
total_filas_consolidado = len(df_consolidado)
print("\nValidacion de la consolidacion")
print(f"Suma de filas de archivos individuales: {total_filas_individuales}")
print(f"Cantidad de filas en el DataFrame consolidado: {total_filas_consolidado}")

if total_filas_individuales == total_filas_consolidado:
    print("Consolidacion Exitosa")
else:
    print("Error en la consolidacion")

display(df_consolidado.head())

Archivos encontrados en la carpeta: 12
(397, 8)

Validacion de la consolidacion
Suma de filas de archivos individuales: 397
Cantidad de filas en el DataFrame consolidado: 397
Consolidacion Exitosa


,fecha,producto,vendedor,region,cantidad,precio_unitario,total,mes
0,2024-04-01,Monitor,Camila Vega,Oriente,2,290000,580000,abril
1,2024-04-02,Disco SSD,Valeria Rios,Norte,2,75000,150000,abril
2,2024-04-03,Teclado,Valeria Rios,Poniente,5,45000,225000,abril
3,2024-04-04,Auriculares,Diego Parra,Poniente,4,68000,272000,abril
4,2024-04-05,Webcam,Camila Vega,Norte,2,55000,110000,abril


Se implementó un bucle for automatizado con pathlib.glob() para indexar los 12 archivos CSV mensuales. Durante la iteración, se extrajo el nombre del mes desde la raíz de cada archivo, se incorporó como una nueva columna y los DataFrames resultantes se almacenaron en una lista. Al cierre, se unificaron las estructuras mediante pd.concat() y se validó matemáticamente la integridad del proceso contrastando el total de filas consolidadas contra la suma de los registros individuales.

In [71]:
#Ejercicio 3: Análisis del consolidado y exportación 
#3.1
tabla_resumen_mensual = df_consolidado.groupby('mes', as_index=False)['ingreso_total'].sum()

orden_meses = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 
               'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']

tabla_resumen_mensual['mes'] = pd.Categorical(tabla_resumen_mensual['mes'], categories=orden_meses, ordered=True)
tabla_resumen_mensual = tabla_resumen_mensual.sort_values('mes').reset_index(drop=True)

print("Resumen ingresos mensuales")
display(tabla_resumen_mensual)
print(tabla_resumen_mensual['ingreso_total'].sum())
ruta_salida = Path('datos/salida')
ruta_salida.mkdir(parents=True, exist_ok=True)

Resumen ingresos mensuales


,mes,ingreso_total
0,enero,5739000
1,febrero,12895000
2,marzo,22336000
3,abril,17987000
4,mayo,19499000
5,junio,22156000
6,julio,24528000
7,agosto,16307000
8,septiembre,17044000
9,octubre,12831000


215803000


In [36]:
#3.2

# Calcular columna auxiliar de ingreso total (Precio * Cantidad)
df_consolidado['ingreso_total'] = df_consolidado['precio_unitario'] * df_consolidado['cantidad']

# Métrica A: Total de ventas por mes
resumen_mensual = df_consolidado.groupby('mes', as_index=False)['ingreso_total'].sum()

# Métrica B: Producto más vendido (por cantidad total unidades)
resumen_productos = df_consolidado.groupby('producto', as_index=False)['cantidad'].sum()
prod_mas_vendido = resumen_productos.sort_values(by='cantidad', ascending=False).iloc[0]

# Métrica C: Mes con mayor ingreso monetario
mes_mayor_ingreso = resumen_mensual.sort_values(by='ingreso_total', ascending=False).iloc[0]

# Métrica D: Promedio de venta por transacción
promedio_transaccion = df_consolidado['ingreso_total'].mean()

#Imprimir metricas
print("Metricas consolidadas")
print(f"Producto más vendido: {prod_mas_vendido['producto']} ({prod_mas_vendido['cantidad']} unidades)")
print(f"Mes con mayor ingreso: {mes_mayor_ingreso['mes']} (${mes_mayor_ingreso['ingreso_total']:.2f} CLP)")
print(f"Promedio por transacción: ${promedio_transaccion:.2f} CLP\n")

#Exportar a CSV en la ruta creada
df_consolidado.to_csv(ruta_salida / 'ventas_2024_consolidado.csv', index=False, encoding='utf-8')


Metricas consolidadas
Producto más vendido: Webcam (176 unidades)
Mes con mayor ingreso: diciembre ($28508000.00 CLP)
Promedio por transacción: $543584.38 CLP



In [51]:
#3.3
#Exportar a Excel con tres pestañas estructuradas
ruta_excel = ruta_salida / 'reporte_anual_2024.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:  
    df_consolidado.to_excel(writer, sheet_name='Datos_Completos', index=False)  
    resumen_mensual.to_excel(writer, sheet_name='Resumen_Mensual', index=False)  
    resumen_productos.to_excel(writer, sheet_name='Resumen_Productos', index=False)  

# 4. Validación estructural del archivo Excel generado
with pd.ExcelFile(ruta_excel) as libro:  
    print("Validacion de hojas en excel")
    print(f"Hojas encontradas en el archivo: {libro.sheet_names}")

Validacion de hojas en excel
Hojas encontradas en el archivo: ['Datos_Completos', 'Resumen_Mensual', 'Resumen_Productos']


In [59]:
#Ejercicio 4: Manejo de errores y casos borde 
#Caso 1 — FileNotFoundError
# Codigo con error
pd.read_csv('datos/ventas/ventas_enero.CSV') # extension en mayusculas
# Correccion: usa .exists() para verificar antes de leer
ruta = Path('___________')
if ruta.exists():
    df = pd.read_csv(ruta)
else:
    print('Archivo no encontrado:', ruta.resolve())

Archivo no encontrado: C:\Users\l410_pc27\Desktop\progracien\___________


In [60]:
#Código corregido 
ruta = Path('datos/ventas/ventas_enero.csv')
if ruta.exists():
    df = pd.read_csv(ruta)
    print(f"Éxito: Archivo cargado correctamente. Shape: {df.shape}")
else:
    print('Archivo no encontrado:', ruta.resolve())

Éxito: Archivo cargado correctamente. Shape: (31, 7)


In [62]:
#Caso 2 — EmptyDataError (glob sin resultados)
#Codigo con error
carpeta_erronea = Path('datos/ventas_2025')
dfs = [pd.read_csv(a) for a in carpeta_erronea.glob('*.csv')]
pd.concat(dfs) # ValueError si dfs esta vacio

ValueError: No objects to concatenate

In [63]:
#Codigo corregido
archivos = list(carpeta_erronea.glob('*.csv'))
if len(archivos) == 0:
    print('No se encontraron archivos en:', carpeta_erronea)
else:
    resultado = pd.concat([pd.read_csv(a) for a in archivos], ignore_index=True)


No se encontraron archivos en: datos\ventas_2025


In [66]:
#Caso 3 — UnicodeDecodeError
# Codigo con error (el archivo usa encoding cp1252)
pd.read_csv('datos/archivo_cp1252.csv') # falla sin encoding correcto

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 17: invalid continuation byte

In [67]:
df_corregido = pd.read_csv('datos/archivo_cp1252.csv', encoding='cp1252')
print("Éxito: Contenido leído correctamente:")
display(df_corregido.head())

Éxito: Contenido leído correctamente:


,nombre,ciudad
0,José,Valparaíso
1,María,Concepción


In [68]:
#Caso 4 — Asignación de columna fuera del loop
# Codigo incorrecto: la columna 'mes' se asigna fuera del loop
dfs = []
for archivo in sorted(Path('datos/ventas').glob('*.csv')):
df = pd.read_csv(archivo)
dfs.append(df)
df['mes'] = archivo.stem # ERROR: solo modifica el ultimo df

IndentationError: expected an indented block after 'for' statement on line 4 (1131081264.py, line 5)

In [70]:
# Código corregido: la columna mes debe asignarse dentro del loop
dfs = []
for archivo in sorted(Path('datos/ventas').glob('*.csv')):  
    df = pd.read_csv(archivo)  
    mes = archivo.stem.split('_')[1]  # Extraer el mes correspondiente
    df['mes'] = mes                   # Corrección: Asignar dentro del ciclo
    dfs.append(df)  

df_final = pd.concat(dfs, ignore_index=True)
print("Frecuencia de registros por columna 'mes' asignada correctamente:")
print(df_final['mes'].value_counts())

Frecuencia de registros por columna 'mes' asignada correctamente:
mes
diciembre     62
agosto        31
enero         31
julio         31
octubre       31
marzo         31
mayo          31
abril         30
noviembre     30
junio         30
septiembre    30
febrero       29
Name: count, dtype: int64
